# MVTec AD Bottle Dataset Audit

This notebook summarizes the file counts and integrity checks for the bottle category. The figures show how ground-truth masks align with visible defects and how much of each original image they cover.

The inventory and measurements come from the saved output of the shared audit script. The original MVTec AD test split is used for this audit and final evaluation. [MVTec AD dataset](https://www.mvtec.com/research-teaching/datasets/mvtec-ad)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

working_dir = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in (working_dir, *working_dir.parents)
        if (folder / "data_tools" / "audit_analysis.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project folder in VS Code and run the notebook again.")

report_path = project_root / "outputs" / "audit" / "bottle.json"

if not report_path.is_file():
    raise FileNotFoundError(
        "Generate the report from the project root with "
        "python data_tools/audit.py --root data/raw/mvtec-ad --category bottle"
    )

bottle_audit = json.loads(report_path.read_text(encoding="utf-8"))
if bottle_audit["category"] != "bottle" or bottle_audit["status"] != "PASS":
    raise ValueError("The notebook requires a passing bottle audit report.")

dataset_root = project_root / bottle_audit["dataset_root"]
mask_areas = pd.DataFrame(bottle_audit["mask_measurements"])
defect_labels = {
    "broken_small": "Small break",
    "broken_large": "Large break",
    "contamination": "Contamination",
}

In [ ]:
print(f"Bottle audit: {bottle_audit['status']}")

inventory = pd.DataFrame(bottle_audit["inventory"]).rename(
    columns={"folder": "Folder", "expected": "Expected PNGs", "actual": "Actual PNGs"}
)
display(inventory)

integrity_summary = pd.DataFrame(
    [
        ("Matched image–mask pairs", bottle_audit["matched_pairs"]),
        ("Missing masks", bottle_audit["missing_masks"]),
        ("Masks without a matching image", bottle_audit["unmatched_masks"]),
        (
            "Exact duplicate groups across train and test",
            bottle_audit["cross_split_duplicate_groups"],
        ),
        ("Integrity issues", len(bottle_audit["issues"])),
    ],
    columns=["Check", "Result"],
)
display(integrity_summary)

## Ground-truth mask overlays

Each row shows the first audited example of a defect type. The original test image is on the left. The red ground-truth mask is overlaid on the right.

In [ ]:
fig, axes = plt.subplots(len(defect_labels), 2, figsize=(8, 10), layout="constrained")

for (defect_type, label), (original_ax, overlay_ax) in zip(defect_labels.items(), axes):
    example = next(
        row for row in bottle_audit["mask_measurements"]
        if row["defect_type"] == defect_type
    )

    with Image.open(dataset_root / example["image_id"]) as source:
        image = np.asarray(source.convert("RGB"))

    with Image.open(dataset_root / example["mask_id"]) as source:
        mask = np.asarray(source.convert("L")) > 0

    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask] = (1.0, 0.15, 0.10, 0.55)

    original_ax.imshow(image)
    original_ax.set_title(f"{label} · Original")
    overlay_ax.imshow(image)
    overlay_ax.imshow(overlay)
    overlay_ax.set_title(f"{label} · Ground truth")

    original_ax.axis("off")
    overlay_ax.axis("off")

fig.suptitle("MVTec AD bottle defect examples")
plt.show()

## Defect size distribution

Defect area is the percentage of original image pixels marked by the ground-truth mask. Each box shows the middle half of the measured areas for one defect type, and the line inside shows the median. The horizontal axis uses a log scale to make small areas easier to compare.

In [ ]:
area_groups = [
    mask_areas.loc[mask_areas["defect_type"] == defect_type, "area_percent"]
    for defect_type in defect_labels
]
plot_labels = [
    f"{label} (n={len(areas)})"
    for label, areas in zip(defect_labels.values(), area_groups)
]

fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
ax.boxplot(
    area_groups,
    orientation="horizontal",
    tick_labels=plot_labels,
    patch_artist=True,
    boxprops={"facecolor": "#cfe3f5", "edgecolor": "#275d86"},
    medianprops={"color": "#123951", "linewidth": 2},
)
ax.set_xscale("log")
ax.set_xlabel("Defect mask area (% of original image, log scale)")
ax.set_title("Defect sizes in MVTec AD bottle")
ax.grid(axis="x", alpha=0.25)
plt.show()

area_summary = (
    mask_areas.groupby("defect_type")["area_percent"]
    .agg(["count", "min", "median", "max"])
    .reindex(list(defect_labels))
    .rename(index=defect_labels)
    .rename_axis("Defect type")
    .rename(columns={
        "count": "Images",
        "min": "Minimum (%)",
        "median": "Median (%)",
        "max": "Maximum (%)",
    })
    .round(4)
)
display(area_summary)

width, height = bottle_audit["resize_size"]
lost_masks = bottle_audit["masks_emptied_after_resize"]
print(f"Masks that became empty at {width} × {height}: {lost_masks} of {len(mask_areas)}")

## Implications for the experiment

In the recorded bottle audit, all 63 defective images had matching masks, and the audit found zero integrity issues. All 63 masks retained at least one marked pixel at 224 × 224 using nearest-neighbor interpolation.

The models will receive 224 × 224 images. The evaluator will compare the predicted anomaly maps with the original-resolution masks. The defect-area measurements will help describe which sizes of defects the models detect or miss.